#### Farriel Arrianta Akbar Pratama / 077

### Kegiatan 1 - Exploratory Data Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', '#FF6692', '#B6E880', '#FF97FF']

df = pd.read_csv('cafe_sales.csv')
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.colheader_justify", "center")
pd.set_option("display.precision", 2)

####  1. Menampilkan metadata dataset secara lengkap disertai penjelasan ringkas

In [2]:
print('=' * 50)
print('METADATA DATASET CAFE SALES')
print('=' * 50)
print(f'Jumlah baris: {df.shape[0]}')
print(f'Jumlah kolom: {df.shape[1]}')
print('\nInformasi kolom:')
print(df.info())
print('\nRingkasan statistik:')
print(df.describe())
print('\nSampel data (5 baris pertama):')
print(df.head())

print('\nPenjelasan Ringkas Dataset:')
print('- Dataset ini berisi data transaksi penjualan kafe')
print('- Kolom Transaction ID berisi ID unik untuk setiap transaksi')
print('- Kolom Item berisi jenis item yang dibeli')
print('- Kolom Quantity berisi jumlah item yang dibeli')
print('- Kolom Price Per Unit berisi harga per satuan item')
print('- Kolom Total Spent berisi total pengeluaran (quantity * price per unit)')
print('- Kolom Payment Method berisi metode pembayaran yang digunakan')
print('- Kolom Location berisi lokasi transaksi (Takeaway atau In-store)')
print('- Kolom Transaction Date berisi tanggal transaksi')

METADATA DATASET CAFE SALES
Jumlah baris: 10000
Jumlah kolom: 8

Informasi kolom:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB
None

Ringkasan statistik:
       Transaction ID  Item  Quantity Price Per Unit Total Spent  Payment Method  Location Transaction Date
count          10000    9667   9862        9821          9827               7421      6735         9841    
unique         10000      10      7           8            19     

####  2. Mengolah data (data cleaning dan data manipulation)

In [ ]:
print('\n' + '=' * 50)
print('DATA CLEANING DAN MANIPULATION')
print('=' * 50)


print(f'\nNilai yang hilang pada setiap kolom:')
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print(f'\nJumlah data duplikat: {duplicate_count}')

print('\nNilai unik pada setiap kolom:')
for col in df.columns:
    if df[col].dtype == 'object':
        print(f"{col}: {df[col].unique()}")

print('\nMasalah pada dataset:')
print('1. Terdapat nilai ERROR dan UNKNOWN pada beberapa kolom')
print('2. Terdapat nilai kosong (empty string) pada beberapa kolom')
print('3. Terdapat nilai yang bermasalah pada kolom Total Spent')

df_clean = df.copy()
df_clean = df_clean.replace(['ERROR', 'UNKNOWN', ''], np.nan)

# atau bisa juga menggunakan :
# df_clean = df_clean.replace(['ERROR', ''], np.nan)


numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')


mask = df_clean['Total Spent'].isna() & df_clean['Quantity'].notna() & df_clean['Price Per Unit'].notna()
df_clean.loc[mask, 'Total Spent'] = df_clean.loc[mask, 'Quantity'] * df_clean.loc[mask, 'Price Per Unit']

df_clean['Transaction Date'] = pd.to_datetime(df_clean['Transaction Date'])

df_clean['Month'] = df_clean['Transaction Date'].dt.month
df_clean['Month_Name'] = df_clean['Transaction Date'].dt.month_name()
df_clean['Day'] = df_clean['Transaction Date'].dt.day_name()

print('\nRingkasan data setelah pembersihan:')
print(f'Jumlah baris awal: {df.shape[0]}')
print(f'Jumlah baris setelah pembersihan: {df_clean.shape[0]}')
print('\nNilai yang hilang setelah pembersihan:')
print(df_clean.isnull().sum())


DATA CLEANING DAN MANIPULATION

Nilai yang hilang pada setiap kolom:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Jumlah data duplikat: 0

Nilai unik pada setiap kolom:
Transaction ID: ['TXN_1961373' 'TXN_4977031' 'TXN_4271903' ... 'TXN_5255387' 'TXN_7695629'
 'TXN_6170729']
Item: ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']
Quantity: ['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]
Price Per Unit: ['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']
Total Spent: ['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']
Payment Method: ['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]
Location: ['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']
Transaction Date: ['2023-09-08' '2023-05-16'

####  3. Menganalisis business problem/question (minimal 5)

In [4]:
print('\n' + '=' * 50)
print('ANALISIS BUSINESS PROBLEM/QUESTION')
print('=' * 50)


ANALISIS BUSINESS PROBLEM/QUESTION


##### Question 1: Bagaimana pola penjualan item menurut bulan?

In [ ]:
print('\nQuestion 1: Bagaimana pola penjualan item menurut bulan?')
monthly_sales = df_clean.groupby('Month_Name')['Total Spent'].sum().reset_index()
monthly_item_count = df_clean.groupby('Month_Name')['Quantity'].sum().reset_index()

print('Penjualan Bulanan:')
print(monthly_sales.sort_values('Total Spent', ascending=False))
print('\nJumlah Item Terjual per Bulan:')
print(monthly_item_count.sort_values('Quantity', ascending=False))

print('\nAnalisis:')
best_month = monthly_sales.iloc[monthly_sales['Total Spent'].argmax()]['Month_Name']
worst_month = monthly_sales.iloc[monthly_sales['Total Spent'].argmin()]['Month_Name']
print(f'- Bulan dengan penjualan tertinggi: {best_month}')
print(f'- Bulan dengan penjualan terendah: {worst_month}')
print(f'- Terdapat pola peningkatan/penurunan penjualan di sepanjang tahun yang dapat dianalisis lebih lanjut')


Question 1: Bagaimana pola penjualan item menurut bulan?
Penjualan Bulanan:
   Month_Name  Total Spent
6        June    7350.0   
10    October    7302.0   
4     January    7242.0   
7       March    7214.5   
2    December    7177.0   
0       April    7168.0   
1      August    7077.5   
9    November    6957.0   
8         May    6941.5   
5        July    6877.5   
11  September    6846.0   
3    February    6633.5   

Jumlah Item Terjual per Bulan:
   Month_Name  Quantity
10    October   2415.0 
7       March   2369.0 
6        June   2357.0 
4     January   2328.0 
2    December   2320.0 
1      August   2297.0 
9    November   2274.0 
5        July   2262.0 
0       April   2249.0 
11  September   2240.0 
8         May   2213.0 
3    February   2164.0 

Analisis:
- Bulan dengan penjualan tertinggi: June
- Bulan dengan penjualan terendah: February
- Terdapat pola peningkatan/penurunan penjualan di sepanjang tahun yang dapat dianalisis lebih lanjut


##### Question 2: Item apa yang paling populer dan menghasilkan pendapatan tertinggi?

In [ ]:
print('\nQuestion 2: Item apa yang paling populer dan menghasilkan pendapatan tertinggi?')
item_popularity = df_clean.groupby('Item')['Quantity'].sum().reset_index().sort_values('Quantity', ascending=False)
item_revenue = df_clean.groupby('Item')['Total Spent'].sum().reset_index().sort_values('Total Spent', ascending=False)

print('Item Berdasarkan Jumlah Terjual:')
print(item_popularity.head())
print('\nItem Berdasarkan Pendapatan:')
print(item_revenue.head())

print('\nAnalisis:')
most_popular = item_popularity.iloc[0]['Item']
most_revenue = item_revenue.iloc[0]['Item']
print(f'- Item yang paling banyak terjual: {most_popular}')
print(f'- Item yang menghasilkan pendapatan tertinggi: {most_revenue}')
print(f'- Terdapat perbedaan antara item yang paling populer dan yang menghasilkan pendapatan tertinggi')
print(f'  yang menunjukkan strategi harga yang berbeda untuk setiap item')



Question 2: Item apa yang paling populer dan menghasilkan pendapatan tertinggi?
Item Berdasarkan Jumlah Terjual:
     Item    Quantity
3     Juice   3373.0 
1    Coffee   3368.0 
0      Cake   3329.0 
4     Salad   3310.0 
5  Sandwich   3245.0 

Item Berdasarkan Pendapatan:
     Item    Total Spent
4     Salad    17320.0  
5  Sandwich    13664.0  
6  Smoothie    13320.0  
3     Juice    10509.0  
0      Cake    10395.0  

Analisis:
- Item yang paling banyak terjual: Juice
- Item yang menghasilkan pendapatan tertinggi: Salad
- Terdapat perbedaan antara item yang paling populer dan yang menghasilkan pendapatan tertinggi
  yang menunjukkan strategi harga yang berbeda untuk setiap item


##### Question 3: Bagaimana perbandingan metode pembayaran yang digunakan oleh pelanggan?

In [ ]:
print('\nQuestion 3: Bagaimana perbandingan metode pembayaran yang digunakan oleh pelanggan?')
payment_count = df_clean['Payment Method'].value_counts()
payment_revenue = df_clean.groupby('Payment Method')['Total Spent'].sum().sort_values(ascending=False)

print('Jumlah Transaksi per Metode Pembayaran:')
print(payment_count)
print('\nPendapatan per Metode Pembayaran:')
print(payment_revenue)

print('\nAnalisis:')
most_used_payment = payment_count.index[0]
highest_revenue_payment = payment_revenue.index[0]
print(f'- Metode pembayaran yang paling sering digunakan: {most_used_payment}')
print(f'- Metode pembayaran dengan pendapatan tertinggi: {highest_revenue_payment}')
print(f'- Terdapat korelasi antara frekuensi penggunaan metode pembayaran dan pendapatan')
print(f'- Hal ini dapat membantu kafe dalam menentukan kebijakan pembayaran yang lebih efektif')


Question 3: Bagaimana perbandingan metode pembayaran yang digunakan oleh pelanggan?
Jumlah Transaksi per Metode Pembayaran:
Payment Method
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

Pendapatan per Metode Pembayaran:
Payment Method
Credit Card       20427.0
Digital Wallet    20383.5
Cash              20366.5
Name: Total Spent, dtype: float64

Analisis:
- Metode pembayaran yang paling sering digunakan: Digital Wallet
- Metode pembayaran dengan pendapatan tertinggi: Credit Card
- Terdapat korelasi antara frekuensi penggunaan metode pembayaran dan pendapatan
- Hal ini dapat membantu kafe dalam menentukan kebijakan pembayaran yang lebih efektif


##### Question 4: Bagaimana perbandingan lokasi transaksi (Takeaway vs In-store)?

In [ ]:
print('\nQuestion 4: Bagaimana perbandingan lokasi transaksi (Takeaway vs In-store)?')
location_count = df_clean['Location'].value_counts()
location_revenue = df_clean.groupby('Location')['Total Spent'].sum().sort_values(ascending=False)

print('Jumlah Transaksi per Lokasi:')
print(location_count)
print('\nPendapatan per Lokasi:')
print(location_revenue)

print('\nAnalisis:')
most_visited = location_count.index[0]
highest_revenue_loc = location_revenue.index[0]
print(f'- Lokasi dengan jumlah transaksi terbanyak: {most_visited}')
print(f'- Lokasi dengan pendapatan tertinggi: {highest_revenue_loc}')
print(f'- Analisis ini dapat membantu kafe dalam menentukan strategi layanan')
print(f'  yang lebih fokus pada lokasi yang lebih menguntungkan')


Question 4: Bagaimana perbandingan lokasi transaksi (Takeaway vs In-store)?
Jumlah Transaksi per Lokasi:
Location
Takeaway    3022
In-store    3017
Name: count, dtype: int64

Pendapatan per Lokasi:
Location
In-store    27127.0
Takeaway    26487.5
Name: Total Spent, dtype: float64

Analisis:
- Lokasi dengan jumlah transaksi terbanyak: Takeaway
- Lokasi dengan pendapatan tertinggi: In-store
- Analisis ini dapat membantu kafe dalam menentukan strategi layanan
  yang lebih fokus pada lokasi yang lebih menguntungkan


##### Question 5: Apakah ada tren penjualan berdasarkan hari dalam seminggu?

In [ ]:
print('\nQuestion 5: Apakah ada tren penjualan berdasarkan hari dalam seminggu?')
# Urutkan hari dalam seminggu
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_sales = df_clean.groupby('Day')['Total Spent'].sum().reindex(day_order)
daily_item_count = df_clean.groupby('Day')['Quantity'].sum().reindex(day_order)

print('Penjualan Harian:')
print(daily_sales)
print('\nJumlah Item Terjual per Hari:')
print(daily_item_count)

print('\nAnalisis:')
best_day = daily_sales.idxmax()
worst_day = daily_sales.idxmin()
weekend_sales = daily_sales[['Saturday', 'Sunday']].sum()
weekday_sales = daily_sales[['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']].sum()
print(f'- Hari dengan penjualan tertinggi: {best_day}')
print(f'- Hari dengan penjualan terendah: {worst_day}')
print(f'- Perbandingan penjualan akhir pekan vs hari kerja: {weekend_sales:.2f} vs {weekday_sales:.2f}')
print(f'- Data ini dapat membantu kafe dalam mengatur jadwal staf dan promosi berdasarkan hari')


Question 5: Apakah ada tren penjualan berdasarkan hari dalam seminggu?
Penjualan Harian:
Day
Monday       12135.0
Tuesday      12030.5
Wednesday    11616.5
Thursday     12394.0
Friday       12317.5
Saturday     12007.5
Sunday       12285.5
Name: Total Spent, dtype: float64

Jumlah Item Terjual per Hari:
Day
Monday       4003.0
Tuesday      3838.0
Wednesday    3785.0
Thursday     3974.0
Friday       3996.0
Saturday     3907.0
Sunday       3985.0
Name: Quantity, dtype: float64

Analisis:
- Hari dengan penjualan tertinggi: Thursday
- Hari dengan penjualan terendah: Wednesday
- Perbandingan penjualan akhir pekan vs hari kerja: 24293.00 vs 60493.50
- Data ini dapat membantu kafe dalam mengatur jadwal staf dan promosi berdasarkan hari


####  4. Menampilkan analisis deskriptif statistik / data outlier

In [ ]:
print('\n' + '=' * 50)
print('ANALISIS DESKRIPTIF STATISTIK & DATA OUTLIER')
print('=' * 50)

print('\nAnalisis Deskriptif Statistik:')
print(df_clean[['Quantity', 'Price Per Unit', 'Total Spent']].describe())

def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

print('\nDeteksi Outlier:')
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    outliers, lower, upper = detect_outliers(df_clean, col)
    print(f'\nOutlier pada {col}:')
    print(f'- Batas bawah: {lower}')
    print(f'- Batas atas: {upper}')
    print(f'- Jumlah outlier: {len(outliers)}')
    if len(outliers) > 0:
        print(f'- Sampel outlier:')
        print(outliers.head())

print('\nAnalisis Korelasi:')
correlation = df_clean[['Quantity', 'Price Per Unit', 'Total Spent']].corr()
print(correlation)

print('\nInterpretasi Korelasi:')
print('- Terdapat korelasi positif yang kuat antara Total Spent dengan Quantity dan Price Per Unit')
print('- Hal ini menunjukkan bahwa peningkatan jumlah item dan harga per unit akan')
print('  berdampak signifikan terhadap total pendapatan')



ANALISIS DESKRIPTIF STATISTIK & DATA OUTLIER

Analisis Deskriptif Statistik:
       Quantity  Price Per Unit  Total Spent
count   9521.00      9467.00       9960.00  
mean       3.03         2.95          8.93  
std        1.42         1.28          6.00  
min        1.00         1.00          1.00  
25%        2.00         2.00          4.00  
50%        3.00         3.00          8.00  
75%        4.00         4.00         12.00  
max        5.00         5.00         25.00  

Deteksi Outlier:

Outlier pada Quantity:
- Batas bawah: -1.0
- Batas atas: 7.0
- Jumlah outlier: 0

Outlier pada Price Per Unit:
- Batas bawah: -1.0
- Batas atas: 7.0
- Jumlah outlier: 0

Outlier pada Total Spent:
- Batas bawah: -8.0
- Batas atas: 24.0
- Jumlah outlier: 268
- Sampel outlier:
    Transaction ID  Item   Quantity  Price Per Unit  Total Spent  Payment Method  Location Transaction Date  Month Month_Name    Day    
10    TXN_2548360   Salad     5.0          5.0          25.0                Cash  Take

### Kegiatan 2 - Data Visualization

In [11]:
print('\n' + '=' * 50)
print('DATA VISUALIZATION')
print('=' * 50)


DATA VISUALIZATION


##### 1. Visualisasi Comparison: Penjualan per Bulan

In [12]:
plt.figure(figsize=(12, 6))
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
monthly_sales_sorted = monthly_sales.set_index('Month_Name').reindex(month_order).reset_index()
plt.bar(monthly_sales_sorted['Month_Name'], monthly_sales_sorted['Total Spent'], color=colors)
plt.title('Penjualan Bulanan', fontsize=16)
plt.xlabel('Bulan', fontsize=12)
plt.ylabel('Total Penjualan', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('monthly_sales.png')
plt.close()

##### 2. Visualisasi Relationship: Scatter Plot Quantity vs Total Spent dengan Item sebagai warna

In [13]:
plt.figure(figsize=(12, 6))
top_items = item_revenue.head(5)['Item'].tolist()
df_top_items = df_clean[df_clean['Item'].isin(top_items)]
sns.scatterplot(data=df_top_items, x='Quantity', y='Total Spent', hue='Item', palette=colors[:5], s=100, alpha=0.7)
plt.title('Hubungan Quantity vs Total Spent untuk 5 Item Teratas', fontsize=16)
plt.xlabel('Quantity', fontsize=12)
plt.ylabel('Total Spent', fontsize=12)
plt.legend(title='Item')
plt.tight_layout()
plt.savefig('quantity_vs_total.png')
plt.close()

##### 3. Visualisasi Distribution: Distribusi Metode Pembayaran berdasarkan Lokasi

In [14]:
plt.figure(figsize=(12, 6))
payment_location = pd.crosstab(df_clean['Payment Method'], df_clean['Location'])
payment_location.plot(kind='bar', stacked=True, color=colors)
plt.title('Distribusi Metode Pembayaran berdasarkan Lokasi', fontsize=16)
plt.xlabel('Metode Pembayaran', fontsize=12)
plt.ylabel('Jumlah Transaksi', fontsize=12)
plt.legend(title='Lokasi')
plt.tight_layout()
plt.savefig('payment_location.png')
plt.close()

<Figure size 1200x600 with 0 Axes>

### Visualisasi Dashboard - Plotly

In [ ]:
def create_dashboard():
    month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    monthly_sales = df_clean.groupby('Month_Name')['Total Spent'].sum().reindex(month_order).reset_index()
    daily_sales = df_clean.groupby('Day')['Total Spent'].sum().reindex(day_order).reset_index()
    item_sales = df_clean.groupby('Item')['Total Spent'].sum().sort_values(ascending=False).head(10).reset_index()
    payment_methods = df_clean.groupby('Payment Method')['Total Spent'].sum().sort_values(ascending=False).reset_index()
    location_sales = df_clean.groupby('Location')['Total Spent'].sum().reset_index()
    
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            'Penjualan Bulanan', 'Penjualan berdasarkan Hari',
            'Top 10 Item berdasarkan Penjualan', 'Metode Pembayaran',
            'Penjualan berdasarkan Lokasi', 'Distribusi Quantity'
        ),
        specs=[
            [{"type": "bar"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "pie"}],
            [{"type": "pie"}, {"type": "histogram"}]
        ],
        vertical_spacing=0.1,
        horizontal_spacing=0.1
    )
    
    # 1. Penjualan Bulanan
    fig.add_trace(
        go.Bar(x=monthly_sales['Month_Name'], y=monthly_sales['Total Spent'], marker=dict(color=colors)),
        row=1, col=1
    )
    
    # 2. Penjualan berdasarkan Hari
    fig.add_trace(
        go.Bar(x=daily_sales['Day'], y=daily_sales['Total Spent'], marker=dict(color=colors[1:8])),
        row=1, col=2
    )
    
    # 3. Top 10 Item berdasarkan Penjualan
    fig.add_trace(
        go.Bar(x=item_sales['Item'], y=item_sales['Total Spent'], marker=dict(color=colors[:10])),
        row=2, col=1
    )
    
    # 4. Metode Pembayaran
    fig.add_trace(
        go.Pie(labels=payment_methods['Payment Method'], values=payment_methods['Total Spent'], hole=.3),
        row=2, col=2
    )
    
    # 5. Penjualan berdasarkan Lokasi
    fig.add_trace(
        go.Pie(labels=location_sales['Location'], values=location_sales['Total Spent'], hole=.3),
        row=3, col=1
    )
    
    # 6. Distribusi Quantity
    fig.add_trace(
        go.Histogram(x=df_clean['Quantity'], marker=dict(color=colors[5])),
        row=3, col=2
    )
    

    fig.update_layout(
        title_text="Dashboard Analisis Penjualan Kafe",
        height=900,
        width=1200,
        showlegend=False
    )
    
    fig.update_xaxes(title_text="Bulan", row=1, col=1)
    fig.update_xaxes(title_text="Hari", row=1, col=2)
    fig.update_xaxes(title_text="Item", row=2, col=1)
    fig.update_xaxes(title_text="Quantity", row=3, col=2)
    
    fig.update_yaxes(title_text="Total Penjualan", row=1, col=1)
    fig.update_yaxes(title_text="Total Penjualan", row=1, col=2)
    fig.update_yaxes(title_text="Total Penjualan", row=2, col=1)
    
    fig.write_html("cafe_sales_dashboard.html")
    
    return fig

dashboard = create_dashboard()